# W&B Reports API — Programmatically Curate a Real KPI Report

Reports let you pin curated runs, charts, and prose into a single shareable page. This notebook builds the narrative view for the real HiLiftAeroML KPI workshop path.

It includes the `tiny-transolver-kpi-train` run from `experiment_tracking.ipynb`; if `sweeps.ipynb` has also run, the same project will contain fine-tune trials and the tuned-checkpoint promotion.

**Prerequisites**
- `wandb login`.
- `experiment_tracking.ipynb` executed end-to-end at least once.
- Optional: run `sweeps.ipynb` first so the report sits downstream of tuned-checkpoint promotion.


## 1. Setup

Imports and the target project. `wandb-workspaces` is a separate
package that wraps the Reports REST API; the workspaces module is
imported as `wr` by convention.


In [ ]:
import os
from pathlib import Path

import wandb
import wandb_workspaces.reports.v2 as wr



def load_local_env(filename: str = ".env") -> None:
    """Load simple KEY=VALUE lines from the nearest .env without echoing secrets."""
    for directory in [Path.cwd(), *Path.cwd().parents]:
        env_path = directory / filename
        if not env_path.exists():
            continue
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip("'\""))
        return


load_local_env()

ENTITY = os.getenv("WANDB_ENTITY", "wandb_emea")
PROJECT = os.getenv("WANDB_PROJECT", "jb_physics_demo")
if "/" in PROJECT:
    PROJECT = PROJECT.split("/")[-1]

print(f"wandb            = {wandb.__version__}")
print(f"target project   = {ENTITY}/{PROJECT}")


## 2. Build the report

A `Report` is a title, a description, and an ordered list of `blocks`.
Two kinds of block matter here:

- **Text blocks** (`H1`, `H2`, `P`, `MarkdownBlock`) — static prose
  authored inline, exactly what you would write in a markdown cell.
- **`PanelGrid`** — the live block. It holds one or more `Runset`s
  (each a query into a specific entity/project, optionally filtered by
  run-name substring) and one or more `Panel`s (`LinePlot`, `BarPlot`,
  `ScatterPlot`, `MediaBrowser`, ...). Panels re-query at render time,
  so runs logged **after** you save the report show up automatically.


In [ ]:
panel_grid = wr.PanelGrid(
    runsets=[
        wr.Runset(
            entity=ENTITY,
            project=PROJECT,
            name="TinyTransolverKpi training",
            query="tiny-transolver-kpi",
        ),
    ],
    panels=[
        wr.LinePlot(
            title="Validation MSE",
            x="epoch",
            y=["val/mse", "baseline/val_mse", "model/selection_score"],
            smoothing_factor=0.3,
        ),
        wr.LinePlot(
            title="Coefficient RMSE",
            x="epoch",
            y=["val/cl_rmse", "val/cd_rmse", "val/cm_rmse"],
        ),
        wr.LinePlot(
            title="TinyTransolverKpi vs pooled-token MLP",
            x="epoch",
            y=["model/selection_score", "baseline/selection_score", "model/selection_margin_vs_baseline"],
        ),
        wr.MediaBrowser(
            title="Dataset visual context",
            media_keys=[
                "dataset/coefficient_trends",
                "dataset/design_variable_heatmap",
                "dataset/semantic_token_map",
                "dataset/high_lift_profile_gallery",
                "dataset/high_lift_profile_matrix",
                "dataset/high_lift_profile_overlay",
                "dataset/featured_profile_card",
                "dataset/profile_flythrough_video",
            ],
            num_columns=3,
            mode="gallery",
        ),
        wr.WeavePanelSummaryTable(table_name="dataset/preview_table"),
        wr.WeavePanelSummaryTable(table_name="dataset/geometry_summary_table"),
        wr.WeavePanelSummaryTable(table_name="dataset/profile_card_table"),
        wr.MediaBrowser(
            title="Profile and slice attribution media",
            media_keys=[
                "model/kpi_profile_slice_video",
                "model/kpi_slice_attribution_video",
                "model/kpi_profile_slice_latest",
                "model/kpi_token_slice_usage_latest",
            ],
            num_columns=2,
            mode="gallery",
        ),
        wr.MediaBrowser(
            title="Prediction evolution videos",
            media_keys=[
                "val/model_baseline_curves_video",
                "val/model_baseline_error_delta_video",
                "val/drag_polar_evolution_video",
            ],
            num_columns=1,
            mode="gallery",
        ),
        wr.MediaBrowser(
            title="Model vs baseline comparison media",
            media_keys=[
                "val/model_baseline_scatter_latest",
                "val/model_baseline_drag_polar_latest",
                "val/model_baseline_curves_latest",
                "val/model_baseline_error_delta_latest",
            ],
            num_columns=2,
            mode="gallery",
        ),
        wr.MediaBrowser(
            title="Prediction media",
            media_keys=[
                "val/predicted_vs_true_latest",
                "val/drag_polar_latest",
                "val/coefficient_curves_latest",
                "val/error_heatmap_latest",
                "val/residual_hist_latest",
            ],
            num_columns=2,
            mode="gallery",
        ),
    ],
)

blocks = [
    wr.H1(text="TinyTransolverKpi — real HiLiftAeroML summary"),
    wr.P(
        text=(
            "Auto-generated summary of the real-data KPI surrogate run. "
            "Panels re-query live at render time, so new training and sweep runs appear here without regenerating the report."
        )
    ),
    wr.H2(text="Coefficient accuracy, visual diagnostics, and baseline guardrail"),
    panel_grid,
    wr.MarkdownBlock(
        text=(
            "**Reading guide.** The primary score is validation MSE on normalised `cl`, `cd`, and `cm`. "
            "Physical RMSE panels are logged in coefficient units. "
            "The pooled-token MLP baseline uses the same token inputs without Physics-Attention, so "
            "`model/selection_margin_vs_baseline` is the quick guardrail check. "
            "The media panels show the real design space, simplified high-lift profiles, semantic tokens, "
            "dataset tables with raw KPI rows, geometry summaries, profile cards, and lightweight videos, "
            "token-to-slice attribution videos over epoch time, explicit model-vs-baseline visual comparisons, "
            "predicted-vs-true scatter, drag polar comparisons, coefficient curves over AoA, residual distributions, "
            "and held-out geometry error heatmaps."
        )
    ),
]


## 3. Save and share

`report.save()` publishes the report to the target project as a draft.
`report.url` returns the shareable link — the same URL you would get
from clicking **Create report** in the W&B UI.


In [ ]:
report = wr.Report(
    entity=ENTITY,
    project=PROJECT,
    title="TinyTransolverKpi — real HiLiftAeroML summary",
    description="Programmatically generated by notebooks/reports.ipynb.",
    blocks=blocks,
)

report.save()

print(report.url)
